# EuroSAT dataset import and shared splits

Load EuroSAT using the same `eurosat_splits.csv` and `eurosat_classes.json` files used by `SqueezeNet_EuroSAT.ipynb`. The train, validation, and test assignments remain identical for later model comparison.

## Configuration

Set `data_dir` to an existing EuroSAT directory, or leave it as `None` to download the dataset through KaggleHub. Set `ROOT` explicitly if the kernel starts outside the repository folder.

In [1]:
from pathlib import Path
from types import SimpleNamespace
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "eurosat_data.py").is_file():
    raise FileNotFoundError(
        "Set ROOT to the repository folder containing eurosat_data.py"
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

args = SimpleNamespace(
    data_dir=None,  # Or Path("/path/to/EuroSAT")
    splits=ROOT / "eurosat_splits.csv",
    classes=ROOT / "eurosat_classes.json",
    
    batch_size=64,
    num_workers=0,
)

## Imports and shared manifest

`load_splits` validates the class order, labels, paths, and exact split counts before images are loaded.

In [2]:
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from eurosat_data import load_splits, validate_images

rows, class_names = load_splits(args.splits, args.classes)
manifest = pd.DataFrame(rows)
NUM_CLASSES = len(class_names)

print(manifest["split"].value_counts())
print("Classes:", class_names)

ImportError: Unable to import required dependencies:
pytz: No module named 'pytz'

## Locate and validate the EuroSAT images

The same KaggleHub dataset source and `EuroSAT` directory layout are used as in the SqueezeNet notebook.

In [ ]:
if args.data_dir is None:
    import kagglehub
    dataset_path = kagglehub.dataset_download("apollo2506/eurosat-dataset")
    DATA_DIR = Path(dataset_path) / "EuroSAT"
else:
    DATA_DIR = Path(args.data_dir)

validate_images(rows, DATA_DIR)
print("Dataset directory:", DATA_DIR)
print("All manifest images found.")

## Create train, validation, and test datasets

Use the same 64 x 64 RGB preprocessing and ImageNet normalization as SqueezeNet. No split is reshuffled or regenerated.

In [ ]:
class EuroSATDataset(Dataset):
    def __init__(self, manifest, split, root, transform):
        self.frame = manifest[manifest["split"] == split].reset_index(drop=True)
        self.root = Path(root)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(self.root / row["relative_path"]) as image:
            image = self.transform(image.convert("RGB"))
        return image, int(row["label"])


image_transform = transforms.Compose([
    transforms.Resize(
        (64, 64),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_dataset = EuroSATDataset(manifest, "train", DATA_DIR, image_transform)
val_dataset = EuroSATDataset(manifest, "val", DATA_DIR, image_transform)
test_dataset = EuroSATDataset(manifest, "test", DATA_DIR, image_transform)

generator = torch.Generator().manual_seed(42)
loader_options = {
    "batch_size": args.batch_size,
    "num_workers": args.num_workers,
    "pin_memory": torch.cuda.is_available(),
}
train_loader = DataLoader(
    train_dataset, shuffle=True, generator=generator, **loader_options
)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)

print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))